# Unit19_Example_Transient_Diffusion | PINN 應用實例：薄膜/板材內的暫態擴散

本 Notebook 以 **TensorFlow/Keras 從零實作一個物理資訊神經網路(PINN)**，求解化工傳遞現象的經典**時間相依偏微分方程(PDE)**：**一維暫態擴散（Fick 第二定律）**。這是 PINN 最具代表性的應用——在**時空域 $(x,t)$** 上直接求解 PDE。

## 目標
- 以 PINN 求解拋物線型 PDE： $\dfrac{\partial C}{\partial t} = D\dfrac{\partial^2 C}{\partial x^2}$ （Fick 第二定律）
- 在**二維時空域**上採樣配點，並同時滿足**初始條件(IC)** 與**兩端邊界條件(BC)**
- **正問題**：已知擴散係數 $D$ ，求解整個時空濃度場 $C(x,t)$ ，與解析解比對
- **反問題**：由稀疏且帶雜訊的量測，**辨識擴散係數** $D$ （薄膜/材料特性鑑定的核心任務）

此問題具有**解析解**可嚴格驗證，資料背景源自擴散理論（Crank, *The Mathematics of Diffusion*）。

---
### 0. 環境設定
- 本範例以解析物理模型產生資料，無需下載外部資料集。
- 使用 TensorFlow/Keras 從零實作 PINN，資料量小，可在 Colab 或本地 CPU 環境執行。

In [1]:
from pathlib import Path
import tensorflow as tf
import os

# ========================================
# 路徑設定 (兼容 Colab 與 Local)
# ========================================
UNIT_OUTPUT_DIR = 'P4_Unit19_Example_Transient_Diffusion'
SOURCE_DATA_DIR = 'transient_diffusion'

try:
    from google.colab import drive
    IN_COLAB = True
    print("✓ 偵測到 Colab 環境，準備掛載 Google Drive...")
    drive.mount('/content/drive', force_remount=True)
except ImportError:
    IN_COLAB = False
    print("✓ 偵測到 Local 環境")

if IN_COLAB:
    shortcut_path = '/content/ChemE-3590'
    try:
        os.remove(shortcut_path)
    except FileNotFoundError:
        pass
    source_path = Path('/content/drive/My Drive/Colab Notebooks/ChemE-3590')
    os.symlink(source_path, shortcut_path)
    NOTEBOOK_DIR = Path(shortcut_path) / 'Part_4' / 'Unit19'
else:
    NOTEBOOK_DIR = Path.cwd()

OUTPUT_DIR = NOTEBOOK_DIR / 'outputs' / UNIT_OUTPUT_DIR
MODEL_DIR = OUTPUT_DIR / 'models'
FIG_DIR = OUTPUT_DIR / 'figs'
for d in (OUTPUT_DIR, MODEL_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"\n✓ Notebook工作目錄: {NOTEBOOK_DIR}")
print(f"✓ 結果輸出目錄: {OUTPUT_DIR}")
print(f"✓ 模型輸出目錄: {MODEL_DIR}")
print(f"✓ 圖檔輸出目錄: {FIG_DIR}")

print(f"\nTensorFlow Version: {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✓ 偵測到 GPU：{gpus[0].name}")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("△ 未偵測到 GPU，將使用 CPU（本範例資料量小，CPU 即可）")

✓ 偵測到 Local 環境

✓ Notebook工作目錄: D:\MyGit\ChemE-3590\Part_4\Unit19
✓ 結果輸出目錄: D:\MyGit\ChemE-3590\Part_4\Unit19\outputs\P4_Unit19_Example_Transient_Diffusion
✓ 模型輸出目錄: D:\MyGit\ChemE-3590\Part_4\Unit19\outputs\P4_Unit19_Example_Transient_Diffusion\models
✓ 圖檔輸出目錄: D:\MyGit\ChemE-3590\Part_4\Unit19\outputs\P4_Unit19_Example_Transient_Diffusion\figs

TensorFlow Version: 2.10.1
✓ 偵測到 GPU：/physical_device:GPU:0


---
### 1. 導入必要套件

In [2]:
# 基礎套件
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# TensorFlow / Keras
from tensorflow import keras
from tensorflow.keras import layers

print(f"TensorFlow版本: {tf.__version__}")
print(f"Keras版本: {keras.__version__}")

# 設定隨機種子以確保結果可重現
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

plt.rcParams['axes.unicode_minus'] = False

TensorFlow版本: 2.10.1
Keras版本: 2.10.0


---
### 2. 問題定義：一維暫態擴散（Fick 第二定律）

考慮溶質（或熱量）在一塊**薄膜 / 平板**內隨時間的擴散。以無因次形式，濃度 $C(x,t)$ 遵循 **Fick 第二定律**（拋物線型 PDE）：

$$
\frac{\partial C}{\partial t} = D\,\frac{\partial^2 C}{\partial x^2}, \qquad x\in[0,1],\ t\in[0,1]
$$

其中 $D$ 是無因次**擴散係數**。本問題需要**一個初始條件**與**兩個邊界條件**：

$$
C(x,0) = \sin(\pi x) \quad (\text{初始濃度分佈}), \qquad C(0,t)=C(1,t)=0 \quad (\text{兩端維持零濃度})
$$

這描述一塊初始帶有正弦濃度分佈的薄膜，兩側表面維持零濃度（例如被大量溶劑沖走），溶質逐漸向兩側擴散流失。此問題有**解析解**：

$$
C(x,t) = \sin(\pi x)\,\exp(-D\pi^2 t)
$$

> 濃度的空間形狀維持正弦，但振幅隨時間**指數衰減**，衰減速率由擴散係數 $D$ 決定。本範例取 $D=0.2$ 。

In [3]:
# 擴散係數真值
D_TRUE = 0.2

def analytical_C(x, t, D=D_TRUE):
    """解析解 C(x,t) = sin(pi*x) * exp(-D*pi^2*t)"""
    return np.sin(np.pi * x) * np.exp(-D * np.pi**2 * t)

# 建立時空網格用於視覺化
nx, nt = 100, 100
xg = np.linspace(0, 1, nx)
tg = np.linspace(0, 1, nt)
XX, TT = np.meshgrid(xg, tg)
C_field = analytical_C(XX, TT)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
cf = axes[0].contourf(XX, TT, C_field, levels=50, cmap='viridis')
fig.colorbar(cf, ax=axes[0], label='Concentration C')
axes[0].set_xlabel('Position  x'); axes[0].set_ylabel('Time  t')
axes[0].set_title('Analytical Space-Time Field  C(x,t)')

for tsnap in [0.0, 0.1, 0.25, 0.5, 1.0]:
    axes[1].plot(xg, analytical_C(xg, tsnap), label=f't={tsnap}')
axes[1].set_xlabel('Position  x'); axes[1].set_ylabel('Concentration C')
axes[1].set_title('Concentration Profiles at Different Times')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'analytical_field.png', dpi=120)
plt.show()

print(f"擴散係數 D = {D_TRUE}")
print(f"衰減因子 exp(-D*pi^2*t): t=0 -> 1.000, t=1 -> {np.exp(-D_TRUE*np.pi**2):.4f}")
print(f"中點濃度 C(0.5, t): t=0 -> {analytical_C(0.5,0):.4f}, t=1 -> {analytical_C(0.5,1):.4f}")

擴散係數 D = 0.2
衰減因子 exp(-D*pi^2*t): t=0 -> 1.000, t=1 -> 0.1389
中點濃度 C(0.5, t): t=0 -> 1.0000, t=1 -> 0.1389


---
### 3. 建立 PINN 模型

與前兩個範例（單一輸入）不同，暫態 PDE 的解 $C(x,t)$ 依賴**兩個自變數**：位置 $x$ 與時間 $t$ 。
因此 PINN 的輸入層有 **2 個節點** $(x, t)$ ，輸出為該時空點的濃度 $C$ 。採用 $\tanh$ 激活以支援二階導數。

In [4]:
def build_pinn(n_hidden=4, n_units=30, seed=SEED):
    """以 (x, t) 為輸入、C 為輸出的 PINN (MLP + tanh)"""
    init = keras.initializers.GlorotNormal(seed=seed)
    inp = keras.Input(shape=(2,), name='x_t')
    h = inp
    for i in range(n_hidden):
        h = layers.Dense(n_units, activation='tanh',
                         kernel_initializer=init, name=f'hidden_{i+1}')(h)
    out = layers.Dense(1, kernel_initializer=init, name='C')(h)
    return keras.Model(inp, out, name='PINN_Transient')

pinn_demo = build_pinn()
pinn_demo.summary()

Model: "PINN_Transient"


_________________________________________________________________


 Layer (type)                Output Shape              Param #   


 x_t (InputLayer)            [(None, 2)]               0         


 hidden_1 (Dense)            (None, 30)                90        


 hidden_2 (Dense)            (None, 30)                930       


... (3 outputs left out)

Total params: 2,911


Trainable params: 2,911


Non-trainable params: 0


_________________________________________________________________


---
### 4. 正問題：時空域殘差與自動微分

**正問題**：已知 $D=0.2$ ，求解整個時空濃度場。PDE 殘差同時用到**對時間的一階導數**與**對空間的二階導數**：

$$
r(x,t) = \frac{\partial \hat{C}_\theta}{\partial t} - D\,\frac{\partial^2 \hat{C}_\theta}{\partial x^2}
$$

以巢狀 `tf.GradientTape` 計算：外層對 $x$ 求導得 $\partial^2 C/\partial x^2$ ，內層（persistent）一次取得 $\partial C/\partial x$ 與 $\partial C/\partial t$ 。

In [5]:
def pde_residual(model, xt, D):
    """時空殘差 r = dC/dt - D * d2C/dx2"""
    x = xt[:, 0:1]
    t = xt[:, 1:2]
    with tf.GradientTape() as g2:
        g2.watch(x)
        with tf.GradientTape(persistent=True) as g1:
            g1.watch(x); g1.watch(t)
            C = model(tf.concat([x, t], axis=1))
        C_x = g1.gradient(C, x)
        C_t = g1.gradient(C, t)
        del g1
    C_xx = g2.gradient(C_x, x)
    return C_t - D * C_xx

# 時空配點(collocation) + IC + BC
N_COLLOC = 2000
rng = np.random.default_rng(SEED)
xt_colloc = tf.convert_to_tensor(
    np.column_stack([rng.uniform(0, 1, N_COLLOC),
                     rng.uniform(0, 1, N_COLLOC)]).astype('float32'))

# 初始條件 t=0: C = sin(pi x)
n_ic = 100
x_ic = np.linspace(0, 1, n_ic)
xt_ic = tf.convert_to_tensor(np.column_stack([x_ic, np.zeros(n_ic)]).astype('float32'))
C_ic = tf.convert_to_tensor(np.sin(np.pi * x_ic).reshape(-1, 1).astype('float32'))

# 邊界條件 x=0, x=1: C = 0
n_bc = 100
t_bc = np.linspace(0, 1, n_bc)
xt_bc0 = tf.convert_to_tensor(np.column_stack([np.zeros(n_bc), t_bc]).astype('float32'))
xt_bc1 = tf.convert_to_tensor(np.column_stack([np.ones(n_bc), t_bc]).astype('float32'))

print(f"時空配點: {N_COLLOC}， IC 點: {n_ic}， 兩端 BC 點: {n_bc} x2")

時空配點: 2000， IC 點: 100， 兩端 BC 點: 100 x2


### 4.1 訓練正問題 PINN

總損失 = PDE 殘差 + 初始條件 + 邊界條件（IC/BC 各加權 10 倍）：

$$
\mathcal{L} = \mathcal{L}_{\mathrm{pde}} + 10\,\mathcal{L}_{\mathrm{ic}} + 10\,\mathcal{L}_{\mathrm{bc}}
$$

時空 PDE 比前兩個範例更難收斂，這裡採用**學習率分段衰減**（1e-3 → 3e-4 → 1e-4）讓訓練後期更穩定。

In [6]:
pinn_fwd = build_pinn()
# 學習率分段衰減：前期快速下降，後期穩定收斂
lr_sched = keras.optimizers.schedules.PiecewiseConstantDecay(
    boundaries=[6000, 9000], values=[1e-3, 3e-4, 1e-4])
opt_fwd = keras.optimizers.Adam(learning_rate=lr_sched)
W_IC, W_BC = 10.0, 10.0

@tf.function
def train_step_forward():
    with tf.GradientTape() as tape:
        r = pde_residual(pinn_fwd, xt_colloc, D_TRUE)
        loss_pde = tf.reduce_mean(tf.square(r))
        loss_ic = tf.reduce_mean(tf.square(pinn_fwd(xt_ic) - C_ic))
        loss_bc = tf.reduce_mean(tf.square(pinn_fwd(xt_bc0))) + \
                  tf.reduce_mean(tf.square(pinn_fwd(xt_bc1)))
        loss = loss_pde + W_IC * loss_ic + W_BC * loss_bc
    grads = tape.gradient(loss, pinn_fwd.trainable_variables)
    opt_fwd.apply_gradients(zip(grads, pinn_fwd.trainable_variables))
    return loss, loss_pde, loss_ic, loss_bc

EPOCHS_FWD = 12000
for ep in range(EPOCHS_FWD + 1):
    loss, lp, lic, lbc = train_step_forward()
    if ep % 1500 == 0:
        print(f"Epoch {ep:5d} | total={float(loss):.3e} | pde={float(lp):.3e} | "
              f"ic={float(lic):.3e} | bc={float(lbc):.3e}")

Epoch     0 | total=5.062e+00 | pde=3.048e-03 | ic=5.034e-01 | bc=2.505e-03


Epoch  1500 | total=4.058e-03 | pde=2.705e-03 | ic=4.692e-05 | bc=8.838e-05


Epoch  3000 | total=1.580e-03 | pde=1.196e-03 | ic=8.754e-06 | bc=2.964e-05


Epoch  4500 | total=8.486e-04 | pde=6.841e-04 | ic=3.409e-06 | bc=1.304e-05


Epoch  6000 | total=2.148e-03 | pde=3.671e-04 | ic=6.322e-05 | bc=1.149e-04


Epoch  7500 | total=3.184e-04 | pde=2.497e-04 | ic=1.559e-06 | bc=5.312e-06


Epoch  9000 | total=1.743e-04 | pde=1.338e-04 | ic=1.061e-06 | bc=2.991e-06


Epoch 10500 | total=1.271e-04 | pde=9.803e-05 | ic=7.547e-07 | bc=2.149e-06


Epoch 12000 | total=7.959e-05 | pde=6.189e-05 | ic=5.147e-07 | bc=1.255e-06


### 4.2 正問題結果：時空場比較

PINN 在**不使用任何濃度資料**下，僅憑 PDE、IC、BC 重建了整個時空濃度場。我們與解析解比對。

In [7]:
grid = np.column_stack([XX.ravel(), TT.ravel()]).astype('float32')
C_pred = pinn_fwd(tf.convert_to_tensor(grid)).numpy().reshape(XX.shape)
C_exact = analytical_C(XX, TT)
rel_l2 = np.linalg.norm(C_pred - C_exact) / np.linalg.norm(C_exact)
max_abs = np.abs(C_pred - C_exact).max()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.3))
cf0 = axes[0].contourf(XX, TT, C_exact, levels=50, cmap='viridis')
fig.colorbar(cf0, ax=axes[0]); axes[0].set_title('Analytical  C(x,t)')
cf1 = axes[1].contourf(XX, TT, C_pred, levels=50, cmap='viridis')
fig.colorbar(cf1, ax=axes[1]); axes[1].set_title(f'PINN  C(x,t)  (rel. $L_2$={rel_l2:.2e})')
cf2 = axes[2].contourf(XX, TT, np.abs(C_pred - C_exact), levels=50, cmap='magma')
fig.colorbar(cf2, ax=axes[2]); axes[2].set_title('Absolute Error')
for ax in axes:
    ax.set_xlabel('Position  x'); ax.set_ylabel('Time  t')
plt.tight_layout()
plt.savefig(FIG_DIR / 'forward_field.png', dpi=120)
plt.show()

# 不同時間的切片比較
fig, ax = plt.subplots(figsize=(7.5, 4.8))
for tsnap, col in zip([0.0, 0.1, 0.25, 0.5, 1.0],
                      ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']):
    ax.plot(xg, analytical_C(xg, tsnap), '-', color=col, label=f'Exact t={tsnap}')
    gg = np.column_stack([xg, np.full_like(xg, tsnap)]).astype('float32')
    ax.plot(xg, pinn_fwd(tf.convert_to_tensor(gg)).numpy().ravel(), '--', color=col, lw=2)
ax.set_xlabel('Position  x'); ax.set_ylabel('Concentration C')
ax.set_title('Snapshots: Analytical (solid) vs PINN (dashed)')
ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'forward_snapshots.png', dpi=120)
plt.show()

print(f"正問題相對 L2 誤差: {rel_l2:.3e}")
print(f"正問題最大絕對誤差: {max_abs:.3e}")

正問題相對 L2 誤差: 1.251e-03
正問題最大絕對誤差: 2.246e-03


---
### 5. 反問題：擴散係數辨識

**反問題**：在材料鑑定中，薄膜/板材的擴散係數 $D$ 往往**未知**，需由**量測的濃度隨時空變化**反推。這是**滲透實驗(permeation test)** 與**材料擴散特性鑑定**的核心任務。

我們假設只能在少數**時空位置** $(x_j, t_j)$ 取得帶雜訊的濃度量測，把 $D$ 設為**可訓練變數**，與網路一起優化。

In [8]:
# 產生時空上稀疏且帶雜訊的量測
rng2 = np.random.default_rng(SEED)
N_DATA = 40
x_data = rng2.uniform(0, 1, N_DATA)
t_data = rng2.uniform(0, 1, N_DATA)
C_data = analytical_C(x_data, t_data) + rng2.normal(0, 0.02, N_DATA)

xt_data = tf.convert_to_tensor(np.column_stack([x_data, t_data]).astype('float32'))
C_data_tf = tf.convert_to_tensor(C_data.reshape(-1, 1).astype('float32'))

fig, ax = plt.subplots(figsize=(7, 5))
cf = ax.contourf(XX, TT, C_exact, levels=50, cmap='viridis', alpha=0.6)
fig.colorbar(cf, ax=ax, label='True C(x,t)')
sc = ax.scatter(x_data, t_data, c=C_data, cmap='viridis',
                edgecolors='red', s=70, linewidths=1.5, label='Noisy measurements')
ax.set_xlabel('Position  x'); ax.set_ylabel('Time  t')
ax.set_title(f'Sparse Noisy Space-Time Measurements (N={N_DATA})')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'inverse_data.png', dpi=120)
plt.show()

print(f"觀測資料點數: {N_DATA}，雜訊標準差: 0.02")

觀測資料點數: 40，雜訊標準差: 0.02


### 5.1 訓練反問題 PINN

初始猜測 $D_{\mathrm{init}}=0.05$ （遠離真值 0.2）。訓練過程中追蹤 $D$ 的收斂。

In [9]:
pinn_inv = build_pinn()
D_var = tf.Variable(0.05, dtype=tf.float32, name='D')   # 待辨識的擴散係數
opt_inv = keras.optimizers.Adam(learning_rate=5e-3)

@tf.function
def train_step_inverse():
    with tf.GradientTape() as tape:
        C_pred = pinn_inv(xt_data)
        loss_data = tf.reduce_mean(tf.square(C_pred - C_data_tf))
        r = pde_residual(pinn_inv, xt_colloc, D_var)
        loss_pde = tf.reduce_mean(tf.square(r))
        loss_ic = tf.reduce_mean(tf.square(pinn_inv(xt_ic) - C_ic))
        loss_bc = tf.reduce_mean(tf.square(pinn_inv(xt_bc0))) + \
                  tf.reduce_mean(tf.square(pinn_inv(xt_bc1)))
        loss = loss_data + loss_pde + W_IC * loss_ic + W_BC * loss_bc
    var_list = pinn_inv.trainable_variables + [D_var]
    grads = tape.gradient(loss, var_list)
    opt_inv.apply_gradients(zip(grads, var_list))
    return loss, loss_data, loss_pde, D_var

EPOCHS_INV = 12000
D_history = []
for ep in range(EPOCHS_INV + 1):
    loss, ld, lp, dv = train_step_inverse()
    if ep % 300 == 0:
        D_history.append((ep, float(dv)))
    if ep % 1500 == 0:
        print(f"Epoch {ep:5d} | total={float(loss):.3e} | data={float(ld):.3e} | "
              f"pde={float(lp):.3e} | D={float(dv):.4f}")

D_pred = float(D_var)
print(f"\n✓ 辨識結果: D_pred = {D_pred:.4f}  (真值 D_true = {D_TRUE})")
print(f"  相對誤差: {abs(D_pred - D_TRUE)/D_TRUE*100:.2f}%")

Epoch     0 | total=5.174e+00 | data=1.112e-01 | pde=3.869e-03 | D=0.0550


Epoch  1500 | total=2.195e-03 | data=3.160e-04 | pde=1.250e-03 | D=0.1936


Epoch  3000 | total=1.344e-03 | data=2.616e-04 | pde=2.416e-04 | D=0.2024


Epoch  4500 | total=5.206e-04 | data=2.130e-04 | pde=8.512e-05 | D=0.2035


Epoch  6000 | total=3.789e-04 | data=2.157e-04 | pde=6.411e-05 | D=0.2046


Epoch  7500 | total=3.621e-04 | data=2.193e-04 | pde=5.198e-05 | D=0.2032


Epoch  9000 | total=3.736e-04 | data=2.163e-04 | pde=3.912e-05 | D=0.2035


Epoch 10500 | total=1.138e-03 | data=2.288e-04 | pde=5.591e-05 | D=0.2041


Epoch 12000 | total=4.490e-04 | data=2.242e-04 | pde=2.993e-05 | D=0.2040

✓ 辨識結果: D_pred = 0.2040  (真值 D_true = 0.2)
  相對誤差: 2.00%


### 5.2 反問題結果： $D$ 收斂與時空場擬合

左圖顯示可訓練擴散係數 $D$ 從初始猜測收斂到真值；右圖比較辨識後 PINN 解與解析解在數個時間的切片。

In [10]:
D_eps = np.array([h[0] for h in D_history])
D_vals = np.array([h[1] for h in D_history])

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(D_eps, D_vals, 'b-o', ms=3, label='Identified D')
axes[0].axhline(D_TRUE, color='k', ls='--', label=f'True D = {D_TRUE}')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('D')
axes[0].set_title('Convergence of Identified Diffusion Coefficient')
axes[0].legend(); axes[0].grid(alpha=0.3)

for tsnap, col in zip([0.0, 0.25, 0.5, 1.0],
                      ['#1f77b4', '#2ca02c', '#d62728', '#9467bd']):
    axes[1].plot(xg, analytical_C(xg, tsnap), '-', color=col, label=f'Exact t={tsnap}')
    gg = np.column_stack([xg, np.full_like(xg, tsnap)]).astype('float32')
    axes[1].plot(xg, pinn_inv(tf.convert_to_tensor(gg)).numpy().ravel(), '--', color=col, lw=2)
axes[1].set_xlabel('Position  x'); axes[1].set_ylabel('Concentration C')
axes[1].set_title('Inverse PINN (dashed) vs Analytical (solid)')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'inverse_result.png', dpi=120)
plt.show()

print(f"最終 D_pred = {D_pred:.4f} | D_true = {D_TRUE}")

最終 D_pred = 0.2040 | D_true = 0.2


---
### 6. 模型與結果儲存

In [11]:
# 儲存正問題與反問題 PINN 權重 (TensorFlow checkpoint 格式)
fwd_path = MODEL_DIR / 'pinn_transient_forward'
inv_path = MODEL_DIR / 'pinn_transient_inverse'
pinn_fwd.save_weights(str(fwd_path))
pinn_inv.save_weights(str(inv_path))

import json
results = {
    'D_true': D_TRUE,
    'D_pred': D_pred,
    'forward_rel_L2': float(rel_l2),
    'forward_max_abs': float(max_abs),
    'D_rel_error_pct': float(abs(D_pred - D_TRUE)/D_TRUE*100),
}
with open(MODEL_DIR / 'transient_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"✓ 正問題權重儲存: {fwd_path.name}")
print(f"✓ 反問題權重儲存: {inv_path.name}")
print(f"✓ 結果儲存: transient_results.json")
print(json.dumps(results, ensure_ascii=False, indent=2))

✓ 正問題權重儲存: pinn_transient_forward
✓ 反問題權重儲存: pinn_transient_inverse
✓ 結果儲存: transient_results.json
{
  "D_true": 0.2,
  "D_pred": 0.20400023460388184,
  "forward_rel_L2": 0.001250526246811759,
  "forward_max_abs": 0.0022457288578152826,
  "D_rel_error_pct": 2.0001173019409126
}


---
### 7. 總結與文獻參考

本 Notebook 以 PINN 求解化工傳遞現象的時間相依 PDE，完成了：

1. **時空域 PDE 建模**：Fick 第二定律 $\dfrac{\partial C}{\partial t}=D\dfrac{\partial^2 C}{\partial x^2}$ ，解析解 $C=\sin(\pi x)e^{-D\pi^2 t}$
2. **二維輸入 PINN**：網路輸入 $(x,t)$ ，在時空域採樣配點
3. **混合導數殘差**：以巢狀 `tf.GradientTape` 同時算 $\partial C/\partial t$ （一階）與 $\partial^2 C/\partial x^2$ （二階）
4. **IC + 兩端 BC**：初始正弦分佈與兩端零濃度條件
5. **正問題**：無資料下重建整個時空濃度場
6. **反問題**：由稀疏帶雜訊的時空量測**辨識擴散係數** $D$

**工程意義**：暫態擴散方程廣泛用於**薄膜滲透、乾燥、吸附/脫附、藥物釋放、熱傳導**等化工程序。PINN 能由**有限的時空感測資料**反推擴散/傳遞係數，是材料特性鑑定與製程數位分身的實用工具。

**與前兩個 PINN 範例的比較**：

| 範例 | 方程類型 | 自變數 | 導數階數 | 待辨識參數 |
|---|---|---|---|---|
| Unit19 概述 | 一階 ODE (IVP) | $t$ | 一階 | 速率常數 $k$ |
| 觸媒擴散-反應 | 二階 ODE (BVP) | $\xi$ | 二階 | Thiele 模數 $\phi$ |
| 暫態擴散（本例） | 拋物線 PDE | $x, t$ | 一階+二階 | 擴散係數 $D$ |

**文獻參考**：
- Crank, J. (1975). *The Mathematics of Diffusion*, 2nd ed. Oxford University Press.
- Bird, R. B., Stewart, W. E., Lightfoot, E. N. *Transport Phenomena*, 2nd ed. Wiley.
- Raissi, M., Perdikaris, P., Karniadakis, G. E. (2019). "Physics-Informed Neural Networks: A Deep Learning Framework for Solving Forward and Inverse Problems Involving Nonlinear Partial Differential Equations." *Journal of Computational Physics*, 378, 686-707.
- Serebrennikova, A., Teubler, R., Hoffellner, L., Leitner, E., Hirn, U., Zojer, K. (2022). "Transport of Organic Volatiles through Paper: Physics-Informed Neural Networks for Solving Inverse and Forward Problems." *Transport in Porous Media*, 145(3), 589-612.
- Karniadakis, G. E., et al. (2021). "Physics-Informed Machine Learning." *Nature Reviews Physics*, 3, 422-440.